 # DuckDB SQL playground for 4 CSV files in a git repository (fixed: no parameters in CREATE VIEW)



 **Goal:** run SQL queries conveniently over 4 CSV files stored in your git repo.



 **Files:**

 - `backend_events.csv` — product usage events (timestamp, user_id, organization_id, event_type, ...)

 - `hubspot_deals.csv` — HubSpot deals/opportunities (stages, amounts, company associations)

 - `hubspot_companies.csv` — HubSpot companies/organizations

 - `hubspot_contacts.csv` — HubSpot contacts (persons) with organization associations

In [6]:
# (Optional) Install dependencies if needed:
# !pip -q install duckdb pandas


In [7]:
from __future__ import annotations

from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [8]:
# --- PATHS ---
REPO_ROOT = Path.cwd()

# If CSVs are in a subfolder (e.g., data/), change this line:
DATA_DIR = REPO_ROOT / "data"

FILES = {
    "backend_events": DATA_DIR / "backend_events.csv",
    "hubspot_deals": DATA_DIR / "hubspot_deals.csv",
    "hubspot_companies": DATA_DIR / "hubspot_companies.csv",
    "hubspot_contacts": DATA_DIR / "hubspot_contacts.csv",
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing CSV files:\n- " + "\n- ".join(missing) +
        "\n\nCheck DATA_DIR and file names."
    )

# Quick size overview
sizes = {name: p.stat().st_size for name, p in FILES.items()}
pd.DataFrame(
    [{"table": k, "path": str(FILES[k]), "size_mb": round(sizes[k] / 1024 / 1024, 2)} for k in FILES]
).sort_values("size_mb", ascending=False)


,table,path,size_mb
0,backend_events,/Users/vitalii/Desktop/unne/bi-technical-chall...,12.96
3,hubspot_contacts,/Users/vitalii/Desktop/unne/bi-technical-chall...,0.05
1,hubspot_deals,/Users/vitalii/Desktop/unne/bi-technical-chall...,0.02
2,hubspot_companies,/Users/vitalii/Desktop/unne/bi-technical-chall...,0.02


 ## 1) Connect to DuckDB and create VIEWs over CSV files



 - `VIEW` reads CSV on demand (no data copy into the DB file)

 - If you want faster repeated queries, materialize into `TABLE`s (see section 6)

In [9]:
DB_PATH = REPO_ROOT / "local.duckdb"  # recommended to add to .gitignore
con = duckdb.connect(str(DB_PATH))

# Useful settings (adjust as needed)
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA enable_progress_bar=true;")
# con.execute("SET memory_limit='2GB';")  # optional


In [10]:
def sql_string_literal(s: str) -> str:
    """
    Return a safe SQL string literal for DuckDB by escaping single quotes.
    """
    return "'" + s.replace("'", "''") + "'"

def to_abs_posix(p: Path) -> str:
    """
    Convert a path to an absolute POSIX-style string.
    This avoids issues with backslashes on Windows and keeps the SQL literal clean.
    """
    return p.resolve().as_posix()


In [11]:
# Create VIEWs (DuckDB infers schema/types via read_csv_auto)
# NOTE: We inline CSV paths because DuckDB cannot use prepared parameters in CREATE VIEW.
for view_name, csv_path in FILES.items():
    csv_abs = to_abs_posix(Path(csv_path))
    con.execute(f"""
        CREATE OR REPLACE VIEW {view_name} AS
        SELECT *
        FROM read_csv_auto({sql_string_literal(csv_abs)}, HEADER=TRUE);
    """)

print("✅ Done. Views created:", ", ".join(FILES.keys()))
print("DB file:", DB_PATH)


✅ Done. Views created: backend_events, hubspot_deals, hubspot_companies, hubspot_contacts
DB file: /Users/vitalii/Desktop/unne/bi-technical-challenge/local.duckdb


 ### Git recommendation

 Add DuckDB artifacts to `.gitignore` so you do not commit local state:



 ```

 local.duckdb

 local.duckdb.wal

 *.duckdb

 ```

 ## 2) Helpers: run SQL, inspect schema, preview rows

In [12]:
def q(sql: str) -> pd.DataFrame:
    """
    Execute SQL and return a pandas DataFrame.
    Note: This helper does not support parameter binding by design, to keep things simple.
    """
    return con.execute(sql).df()

def describe(table_or_view: str) -> pd.DataFrame:
    return q(f"DESCRIBE {table_or_view}")

def head(table_or_view: str, n: int = 5) -> pd.DataFrame:
    return q(f"SELECT * FROM {table_or_view} LIMIT {n}")

def rowcount(table_or_view: str) -> int:
    return int(con.execute(f"SELECT COUNT(*) FROM {table_or_view}").fetchone()[0])


In [13]:
# Schema for all four objects
schemas = {t: describe(t) for t in FILES.keys()}
for t, df in schemas.items():
    print(f"\n=== {t} ===")
    display(df)



=== backend_events ===


,column_name,column_type,null,key,default,extra
0,event_id,VARCHAR,YES,None,None,None
1,event_name,VARCHAR,YES,None,None,None
2,event_timestamp,TIMESTAMP,YES,None,None,None
3,user_id,VARCHAR,YES,None,None,None
4,organization_id,VARCHAR,YES,None,None,None
5,event_properties,VARCHAR,YES,None,None,None



=== hubspot_deals ===


,column_name,column_type,null,key,default,extra
0,deal_id,BIGINT,YES,None,None,None
1,deal_name,VARCHAR,YES,None,None,None
2,pipeline,VARCHAR,YES,None,None,None
3,is_closed,BOOLEAN,YES,None,None,None
4,is_closed_won,BOOLEAN,YES,None,None,None
5,amount,BIGINT,YES,None,None,None
6,close_date,TIMESTAMP,YES,None,None,None
7,create_date,TIMESTAMP,YES,None,None,None
8,hubspot_company_id,BIGINT,YES,None,None,None
9,deal_type,VARCHAR,YES,None,None,None



=== hubspot_companies ===


,column_name,column_type,null,key,default,extra
0,company_id,BIGINT,YES,None,None,None
1,company_name,VARCHAR,YES,None,None,None
2,domain,VARCHAR,YES,None,None,None
3,industry,VARCHAR,YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,number_of_employees,VARCHAR,YES,None,None,None
6,create_date,TIMESTAMP,YES,None,None,None



=== hubspot_contacts ===


,column_name,column_type,null,key,default,extra
0,contact_id,BIGINT,YES,None,None,None
1,first_name,VARCHAR,YES,None,None,None
2,last_name,VARCHAR,YES,None,None,None
3,email,VARCHAR,YES,None,None,None
4,job_title,VARCHAR,YES,None,None,None
5,hubspot_company_id,BIGINT,YES,None,None,None
6,lifecycle_stage,VARCHAR,YES,None,None,None
7,create_date,TIMESTAMP,YES,None,None,None


In [14]:
# Row counts (via the VIEWs)
counts = [{"table": t, "rows": rowcount(t)} for t in FILES.keys()]
pd.DataFrame(counts).sort_values("rows", ascending=False)


,table,rows
0,backend_events,39206
3,hubspot_contacts,489
2,hubspot_companies,253
1,hubspot_deals,89


In [15]:
# Row samples
for t in FILES.keys():
    print(f"\n--- {t} (head) ---")
    display(head(t, 5))



--- backend_events (head) ---


,event_id,event_name,event_timestamp,user_id,organization_id,event_properties
0,907ca6c5-a6f7-4b9d-bb00-e1ac41296202,UserCreated,2024-07-03 09:42:25,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-23541..."
1,4eb6b45f-1a1a-4b5c-aab0-7b83148fd8a7,OrganizationCreated,2024-07-03 09:42:25,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""organization"": {""id"": ""1dedfc50-1e5b-4624-8f..."
2,e5a7e460-7607-4b4b-8bd1-bdee53e9baaa,SearchUpdated,2024-07-03 13:33:52,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-23541..."
3,14b216e9-dc98-4762-a395-8fcb54325bfe,SearchExecuted,2024-07-05 18:27:49,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-23541..."
4,0cb57b2f-8caf-4228-b98a-07e923393f20,SearchResultAppraised,2024-07-05 23:42:38,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-23541..."



--- hubspot_deals (head) ---


,deal_id,deal_name,pipeline,is_closed,is_closed_won,amount,close_date,create_date,hubspot_company_id,deal_type,currency,date_entered_pre_pitch,date_entered_pitching,date_entered_product_testing,date_entered_price_offering,date_entered_contract_negotiation,date_entered_closed_won,date_entered_closed_lost
0,20000000,Acumed ApS - Annual License,Sales Pipeline,True,True,12000,2025-09-29 01:47:03,2025-08-25 01:47:03,1000000,newbusiness,EUR,NaT,2025-08-25 01:47:03,2025-09-15 01:47:03,NaT,2025-09-22 01:47:03,2025-09-28 01:47:03,NaT
1,20000256,Acumed ApS - Expansion,Sales Pipeline,False,False,8000,NaT,2025-10-22 20:02:47,1000000,existing_business,EUR,2025-10-22 20:02:47,2025-12-09 20:02:47,2025-12-30 20:02:47,NaT,NaT,NaT,NaT
2,20000420,BioVance d.o.o. - Annual License,Sales Pipeline,True,True,7500,2025-09-26 17:05:48,2025-05-24 17:05:48,1000106,newbusiness,EUR,2025-05-24 17:05:48,2025-08-14 17:05:48,2025-08-25 17:05:48,2025-09-14 17:05:48,NaT,2025-09-21 17:05:48,NaT
3,20000604,BioVance d.o.o. - Upsell,Sales Pipeline,True,True,10000,2026-01-06 17:05:48,2025-12-20 17:05:48,1000106,existing_business,EUR,2025-12-20 17:05:48,2025-12-23 17:05:48,2025-12-27 17:05:48,NaT,2026-01-01 17:05:48,2026-01-02 17:05:48,NaT
4,20000884,ClearPath S.A. - Annual License,Sales Pipeline,True,True,8000,2024-09-17 23:01:51,2024-05-06 23:01:51,1000379,newbusiness,EUR,NaT,2024-05-06 23:01:51,2024-05-24 23:01:51,NaT,2024-06-02 23:01:51,2024-06-06 23:01:51,NaT



--- hubspot_companies (head) ---


,company_id,company_name,domain,industry,country,number_of_employees,create_date
0,1000000,Acumed ApS,acumed.com,Dental,Switzerland,11-50,2025-05-05 21:19:33
1,1000106,BioVance d.o.o.,biovance.com,Healthcare IT,Germany,501-1000,2025-05-05 11:38:37
2,1000379,ClearPath S.A.,clearpath.com,Dental,Sweden,501-1000,2024-07-13 19:38:57
3,1000486,DentalPro GmbH,dentalpro.co,Healthcare IT,United Kingdom,11-50,2025-04-24 05:18:11
4,1000636,EvoMed SAS,evomed.co,Medical Devices,Norway,11-50,2025-06-05 21:49:59



--- hubspot_contacts (head) ---


,contact_id,first_name,last_name,email,job_title,hubspot_company_id,lifecycle_stage,create_date
0,500000,Sarah,Bernard,sarah.bernard@acumed.com,Post-Market Surveillance Manager,1000000,customer,2025-10-03 05:02:36
1,500001,Hannah,Lange,hannah.lange@acumed.com,Director Regulatory Affairs,1000000,customer,2025-10-10 15:20:33
2,500011,Yasmin,Schulz,yasmin.schulz@acumed.com,QA/RA Engineer,1000000,customer,2025-06-04 02:14:01
3,500019,Sophie,Lehmann,sophie.lehmann@biovance.com,Compliance Officer,1000106,customer,2025-09-04 05:31:53
4,500027,Florian,Zimmermann,florian.zimmermann@biovance.com,Head of Quality Management,1000106,customer,2025-09-17 21:58:25


 ## 3) Heuristic detection of common key columns (to speed up querying)



 HubSpot exports and backend logs often use different column names.

 The code below tries to auto-detect columns such as `timestamp`, `org_id`, `user_id`, `event_type`, etc.



 If any detected fields are `None`, inspect the `DESCRIBE ...` output above and set them manually.

In [16]:
def get_columns(table: str) -> list[str]:
    df = describe(table)
    return df["column_name"].tolist()

def pick_col(table: str, candidates: list[str]) -> str | None:
    cols = get_columns(table)
    lower_map = {c.lower(): c for c in cols}

    # Exact match (case-insensitive)
    for cand in candidates:
        key = cand.lower()
        if key in lower_map:
            return lower_map[key]

    # Substring match (e.g., "org" matches "organization_id")
    for cand in candidates:
        key = cand.lower()
        for c in cols:
            if key in c.lower():
                return c

    return None

def common_cols(a: str, b: str) -> list[tuple[str, str]]:
    """
    Find case-insensitive intersection of column names.
    Returns pairs (col_in_a, col_in_b).
    """
    ac = get_columns(a)
    bc = get_columns(b)
    amap = {c.lower(): c for c in ac}
    bmap = {c.lower(): c for c in bc}
    inter = sorted(set(amap.keys()) & set(bmap.keys()))
    return [(amap[k], bmap[k]) for k in inter]


In [17]:
# Candidate names (common patterns in exports/logs)
backend_ts = pick_col("backend_events", ["timestamp", "event_timestamp", "created_at", "occurred_at", "time", "ts", "datetime"])
backend_event_type = pick_col("backend_events", ["event_type", "type", "event", "name", "action"])
backend_user_id = pick_col("backend_events", ["user_id", "userid", "user", "actor_id"])
backend_org_id = pick_col("backend_events", ["organization_id", "org_id", "org", "company_id", "account_id", "workspace_id", "tenant_id"])

deal_stage = pick_col("hubspot_deals", ["dealstage", "stage"])
deal_amount = pick_col("hubspot_deals", ["amount", "value", "deal_amount"])
deal_close_date = pick_col("hubspot_deals", ["closedate", "close_date", "close", "won_date", "date_closed"])
deal_org_id = pick_col("hubspot_deals", ["company_id", "associatedcompanyid", "org_id", "organization_id", "hs_company_id", "company"])

org_id = pick_col("hubspot_companies", ["hs_object_id", "company_id", "id", "org_id", "organization_id"])
org_name = pick_col("hubspot_companies", ["name", "company_name", "org_name"])

contact_id = pick_col("hubspot_contacts", ["hs_object_id", "contact_id", "id"])
contact_email = pick_col("hubspot_contacts", ["email", "email_address"])
contact_org_id = pick_col("hubspot_contacts", ["company_id", "associatedcompanyid", "org_id", "organization_id", "hs_company_id"])

detected = pd.DataFrame(
    [
        {"table": "backend_events", "timestamp": backend_ts, "event_type": backend_event_type, "user_id": backend_user_id, "org_id": backend_org_id},
        {"table": "hubspot_deals", "stage": deal_stage, "amount": deal_amount, "close_date": deal_close_date, "org_id": deal_org_id},
        {"table": "hubspot_companies", "org_id": org_id, "org_name": org_name},
        {"table": "hubspot_contacts", "contact_id": contact_id, "email": contact_email, "org_id": contact_org_id},
    ]
)
detected


,table,timestamp,event_type,user_id,org_id,stage,amount,close_date,org_name,contact_id,email
0,backend_events,event_timestamp,event_id,user_id,organization_id,NaN,NaN,NaN,NaN,NaN,NaN
1,hubspot_deals,NaN,NaN,NaN,hubspot_company_id,NaN,amount,close_date,NaN,NaN,NaN
2,hubspot_companies,NaN,NaN,NaN,company_id,NaN,NaN,NaN,company_name,NaN,NaN
3,hubspot_contacts,NaN,NaN,NaN,hubspot_company_id,NaN,NaN,NaN,NaN,contact_id,email


In [18]:
# Check overlapping columns to spot obvious join keys
print("Common columns: hubspot_deals ↔ hubspot_companies")
display(pd.DataFrame(common_cols("hubspot_deals", "hubspot_companies"), columns=["deals_col", "org_col"]))

print("Common columns: hubspot_contacts ↔ hubspot_companies")
display(pd.DataFrame(common_cols("hubspot_contacts", "hubspot_companies"), columns=["contacts_col", "org_col"]))

print("Common columns: backend_events ↔ hubspot_companies")
display(pd.DataFrame(common_cols("backend_events", "hubspot_companies"), columns=["events_col", "org_col"]))


Common columns: hubspot_deals ↔ hubspot_companies


,deals_col,org_col
0,create_date,create_date


Common columns: hubspot_contacts ↔ hubspot_companies


,contacts_col,org_col
0,create_date,create_date


Common columns: backend_events ↔ hubspot_companies


,events_col,org_col


 ## 4) Quick profiling queries (based on detected columns)



 These will run only if the key columns were detected successfully.

 Otherwise, use `DESCRIBE` to find the correct column names and update the variables above.

In [19]:
# Backend events: time range and top event types
if backend_ts and backend_event_type:
    display(q(f"""
        SELECT
          MIN({backend_ts}) AS min_ts,
          MAX({backend_ts}) AS max_ts,
          COUNT(*) AS n_events
        FROM backend_events;
    """))

    display(q(f"""
        SELECT {backend_event_type} AS event_type, COUNT(*) AS n
        FROM backend_events
        GROUP BY 1
        ORDER BY n DESC
        LIMIT 50;
    """))
else:
    print("⚠️ Could not detect timestamp and/or event_type in backend_events. Inspect DESCRIBE backend_events and set column names manually.")


,min_ts,max_ts,n_events
0,2024-07-03 09:42:25,2026-02-08 23:58:52,39206


,event_type,n
0,e21ba2b6-184a-4406-88c9-158035e73974,1
1,92efef8c-86cc-4ce3-bddd-0139296ca81b,1
2,89778000-b9b4-4d25-baea-7732319b7118,1
3,d4c48629-5a36-46c9-8543-3bf61a552978,1
4,9e9d6775-2674-40d2-b8e9-c4535f003f34,1
5,16f01224-e787-46b3-8030-cca561131edc,1
6,e10b74cb-c2e0-4244-b562-fb4203d014b3,1
7,1f0a0a48-f079-4035-9492-36258da01a55,1
8,0e2dd9de-3c8e-46a5-872a-c92b1f75ffd3,1
9,3de53098-97e4-4368-b5ca-4b09d3aba99c,1


In [20]:
# Backend events: top orgs by event volume
if backend_org_id:
    display(q(f"""
        SELECT {backend_org_id} AS org_id, COUNT(*) AS n_events
        FROM backend_events
        GROUP BY 1
        ORDER BY n_events DESC
        LIMIT 50;
    """))
else:
    print("⚠️ Could not detect org_id/organization_id in backend_events. Inspect DESCRIBE backend_events.")


,org_id,n_events
0,ed31a0a1-1ffc-4ef6-aa35-12645225338e,4086
1,1dedfc50-1e5b-4624-8fd1-c689de270bbc,3704
2,ee58d325-cae4-422f-b359-90dbcba18cb3,2826
3,cc55b936-0090-41a1-a09d-6e3ba637c0a6,2705
4,8fc5002e-4c09-4169-8df5-4d9a950961d3,2653
5,ee6d01c8-f3c8-49e5-9b27-9540f7fc7c37,2172
6,839ec0f5-7c99-48ca-9a57-305028d7fa03,1965
7,84ef39a8-5176-4c57-b3fc-4e49ffca3890,1892
8,a7e01074-ebc8-4820-9bcd-00e14ffbde63,1713
9,3c08f8f7-d293-4a5c-acfe-88ebc9de21d5,1643


In [21]:
# Deals: distribution by stage and amount stats (if detected)
if deal_stage:
    display(q(f"""
        SELECT {deal_stage} AS stage, COUNT(*) AS n_deals
        FROM hubspot_deals
        GROUP BY 1
        ORDER BY n_deals DESC
        LIMIT 50;
    """))
else:
    print("⚠️ Could not detect stage/dealstage in hubspot_deals.")

if deal_amount:
    display(q(f"""
        SELECT
          COUNT(*) AS n_deals,
          SUM(TRY_CAST({deal_amount} AS DOUBLE)) AS total_amount,
          AVG(TRY_CAST({deal_amount} AS DOUBLE)) AS avg_amount
        FROM hubspot_deals;
    """))
else:
    print("⚠️ Could not detect amount/value in hubspot_deals (it may have a different name).")


⚠️ Could not detect stage/dealstage in hubspot_deals.


,n_deals,total_amount,avg_amount
0,89,1100000.0,12359.550562


 ## 5) Example JOIN: deals ↔ orgs ↔ backend usage events



 The query below assumes there is a shared organization/company key across:

 - `hubspot_deals` (deal_org_id)

 - `hubspot_companies` (org_id)

 - `backend_events` (backend_org_id)



 If your HubSpot company ID does **not** match your internal organization ID, you will need a mapping table.

In [22]:
if deal_org_id and org_id and backend_org_id and backend_ts:
    sql = f"""
    WITH events_by_org AS (
      SELECT
        {backend_org_id} AS org_id,
        DATE_TRUNC('month', TRY_CAST({backend_ts} AS TIMESTAMP)) AS month,
        COUNT(*) AS n_events
      FROM backend_events
      GROUP BY 1, 2
    ),
    deals_enriched AS (
      SELECT
        d.*,
        {"o." + org_name if org_name else "NULL"} AS org_name
      FROM hubspot_deals d
      LEFT JOIN hubspot_companies o
        ON d.{deal_org_id} = o.{org_id}
    )
    SELECT
      de.{deal_org_id} AS org_id,
      de.org_name,
      {"de." + deal_stage + " AS deal_stage," if deal_stage else ""}
      {"TRY_CAST(de." + deal_amount + " AS DOUBLE) AS deal_amount," if deal_amount else ""}
      e.month,
      e.n_events
    FROM deals_enriched de
    LEFT JOIN events_by_org e
      ON de.{deal_org_id} = e.org_id
    ORDER BY e.month DESC NULLS LAST
    LIMIT 200;
    """
    # Light cleanup in case optional columns are missing
    sql = sql.replace(",\n      \n", "\n").replace(",\n      e.month", "\n      e.month")
    display(q(sql))
else:
    print("⚠️ Not enough columns detected for the JOIN example. Check the `detected` table above and set the variables manually.")
    print("   Required: deal_org_id, org_id, backend_org_id, backend_ts (optional: stage/amount/org_name).")


ParserException: Parser Error: syntax error at or near "TRY_CAST"

LINE 21:       TRY_CAST(de.amount AS DOUBLE) AS deal_amount
               ^

 ## 6) Materialization (optional): speed up repeated queries



 If CSVs are large and you run many queries, you can load them into DuckDB tables once.

 This copies data into `local.duckdb` but makes repeated querying faster.

In [ ]:
# Uncomment to materialize:
# for t in FILES.keys():
#     con.execute(f"DROP TABLE IF EXISTS {t}_tbl;")
#     con.execute(f"CREATE TABLE {t}_tbl AS SELECT * FROM {t};")
# print("✅ Materialized tables created: backend_events_tbl, hubspot_deals_tbl, hubspot_companies_tbl, hubspot_contacts_tbl")


 ## 7) Keeping SQL queries in the repository



 Store queries as `queries/*.sql` and execute them from the notebook.

In [23]:
def run_sql_file(path: Path) -> pd.DataFrame:
    """
    Load a .sql file and execute it.
    """
    sql_text = path.read_text(encoding="utf-8")
    return q(sql_text)

# Example usage:
# sql_path = REPO_ROOT / "queries" / "qa_events.sql"
# display(run_sql_file(sql_path))


 ## 8) Close the connection

 Optional in notebooks, but good practice.

In [ ]:
# con.close()